In [1]:
!pip install -q ultralytics ipywidgets pillow psutil opencv-python-headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.4 MB/s eta 0:00:00a 0:00:01


In [2]:
import cv2
import time
import threading
import psutil
from collections import Counter

from ultralytics import YOLO
from PIL import Image
from IPython.display import display, clear_output
import ipywidgets as widgets


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
import os
import yaml

# -----------------------------
# CONFIG (KAGGLE-SAFE)
# -----------------------------
DATASET_ROOT = "/kaggle/input/vehicle-dataset-for-yolo/vehicle dataset"
CLASSES_FILE = os.path.join(DATASET_ROOT, "classes.txt")

OUTPUT_YAML = "/kaggle/working/data.yaml"  # ✅ writable

# -----------------------------
# Read classes.txt
# -----------------------------
with open(CLASSES_FILE, "r") as f:
    classes = [line.strip() for line in f if line.strip()]

# -----------------------------
# Build YAML
# -----------------------------
data_yaml = {
    "path": DATASET_ROOT,
    "train": "train/images",
    "val": "valid/images",
    "nc": len(classes),
    "names": {i: name for i, name in enumerate(classes)}
}

# -----------------------------
# Write data.yaml
# -----------------------------
with open(OUTPUT_YAML, "w") as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print("✅ data.yaml written to:", OUTPUT_YAML)
print("Classes:", classes)


✅ data.yaml written to: /kaggle/working/data.yaml
Classes: ['car', 'threewheel', 'bus', 'truck', 'motorbike', 'van']


In [4]:
from ultralytics import YOLO

model = YOLO("yolo26s.pt")

model.train(
    data="/kaggle/working/data.yaml",
    epochs=20,
    imgsz=640,
    batch=8,
    name="regnum_train4"
)


Ultralytics 8.4.18 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=regnum_train4, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspecti

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7d416f09e930>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
     

In [6]:
results = model.val(data="/kaggle/working/data.yaml", imgsz=640)

# This returns a dictionary with all metrics
metrics = results.results_dict
print(metrics)


Ultralytics 8.4.18 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 124.2±107.7 MB/s, size: 89.8 KB)
val: Scanning /kaggle/input/vehicle-dataset-for-yolo/vehicle dataset/valid/labels... 899 images, 0 backgrounds, 1 corrupt: 100% ━━━━━━━━━━━━ 900/900 760.8it/s 1.2s0.1s
val: /kaggle/input/vehicle-dataset-for-yolo/vehicle dataset/valid/images/car55.jpg: ignoring corrupt image/label: [Errno 30] Read-only file system: '/kaggle/input/vehicle-dataset-for-yolo/vehicle dataset/valid/images/car55.jpg'
WARNING ⚠️ val: Cache directory /kaggle/input/vehicle-dataset-for-yolo/vehicle dataset/valid is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 57/57 5.7it/s 10.0s.2s
                   all        899       1149      0.973      0.927      0.978      0.922
                   car        181        200      0.969       0.91      0.966      0.941
   

In [7]:
import psutil
import torch

# CPU usage
cpu_percent = psutil.cpu_percent(interval=1)
print(f"CPU Usage: {cpu_percent}%")

# RAM usage
ram = psutil.virtual_memory()
print(f"RAM Usage: {ram.percent}%")

# GPU usage (if using CUDA)
if torch.cuda.is_available():
    print(f"GPU Memory Usage: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"GPU Utilization: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")


CPU Usage: 0.5%
RAM Usage: 18.7%
GPU Memory Usage: 0.31 GB
GPU Utilization: 3.69 GB


In [10]:
from ultralytics import YOLO
import cv2
import time
from collections import Counter

# -------------------------
# Load trained model
# -------------------------
# Export to TensorRT once (can be skipped if already exported)
model = YOLO("/kaggle/working/runs/detect/regnum_train4/weights/best.pt")
model.export(format="engine")

# Load TensorRT engine for inference
model_trt = YOLO("/kaggle/working/runs/detect/regnum_train4/weights/best.engine")
# -------------------------
# Video IO
# -------------------------
cap = cv2.VideoCapture("/kaggle/input/video-3/Video2/Inference -1.mp4")

w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps_in = cap.get(cv2.CAP_PROP_FPS)

out = cv2.VideoWriter(
    "/kaggle/working/regnum_output2.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps_in,
    (w, h)
)

# -------------------------
# Tracking memory
# -------------------------
track_memory = {}        # track_id -> {votes, best_conf}
vehicle_counts = {}      # final per-class counts

# -------------------------
# VERY PERMISSIVE confidence (DO NOT FILTER EARLY)
# -------------------------
MIN_CONF = {
    "car": 0.15,
    "threewheel": 0.10,
    "bus": 0.20,
    "truck": 0.20,
    "motorbike": 0.05,
    "van": 0.15
}

prev_time = time.time()

# -------------------------
# Main loop
# -------------------------
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # -------------------------
    # YOLO + ByteTrack
    # -------------------------
    results = model.track(
        frame,
        conf=0.05,            # LOW to allow weak detections
        iou=0.6,
        imgsz=960,            # helps small vehicles
        max_det=300,
        persist=True,
        tracker="bytetrack.yaml",
        agnostic_nms=True,    # IMPORTANT for mixed vehicles
        verbose=False
    )

    # -------------------------
    # Process detections
    # -------------------------
    if results and results[0].boxes is not None:
        for box in results[0].boxes:

            if box.id is None:
                continue

            track_id = int(box.id)
            cls_id = int(box.cls)
            class_name = model.names[cls_id]
            conf = float(box.conf)

            # Per-class confidence gate (SOFT)
            if class_name in MIN_CONF and conf < MIN_CONF[class_name]:
                continue

            x1, y1, x2, y2 = map(int, box.xyxy[0])

            # -------------------------
            # Track-level class voting
            # -------------------------
            if track_id not in track_memory:
                track_memory[track_id] = {
                    "votes": [],
                    "best_conf": conf
                }
                vehicle_counts[class_name] = vehicle_counts.get(class_name, 0) + 1

            track_memory[track_id]["votes"].append(class_name)
            track_memory[track_id]["best_conf"] = max(
                track_memory[track_id]["best_conf"], conf
            )

            final_class = Counter(
                track_memory[track_id]["votes"]
            ).most_common(1)[0][0]

            conf = track_memory[track_id]["best_conf"]

            # -------------------------
            # Draw box
            # -------------------------
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 3)

            label = f"{final_class} | ID:{track_id} | {conf:.2f}"

            cv2.putText(frame, label, (x1, y1 - 8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 4, cv2.LINE_AA)
            cv2.putText(frame, label, (x1, y1 - 8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)

    # -------------------------
    # FPS
    # -------------------------
    fps = 1.0 / (time.time() - prev_time)
    prev_time = time.time()

    cv2.putText(frame, f"FPS: {fps:.2f}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 0), 2)

    # -------------------------
    # Counts
    # -------------------------
    y = 70
    for cls, count in vehicle_counts.items():
        cv2.putText(frame, f"{cls}: {count}", (10, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 2)
        y += 35

    out.write(frame)

# -------------------------
# Cleanup
# -------------------------
cap.release()
out.release()

print("✅ Saved output: /kaggle/working/regnum_output.mp4")


WARNING ⚠️ TensorRT requires GPU export, automatically assigning device=0
Ultralytics 8.4.18 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
YOLO26s summary (fused): 122 layers, 9,467,502 parameters, 0 gradients, 20.5 GFLOPs

PyTorch: starting from '/kaggle/working/runs/detect/regnum_train4/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (19.4 MB)

ONNX: starting export with onnx 1.20.1 opset 20...
ONNX: slimming with onnxslim 0.1.86...
ONNX: export success ✅ 1.7s, saved as '/kaggle/working/runs/detect/regnum_train4/weights/best.onnx' (36.4 MB)
requirements: Ultralytics requirement ['tensorrt-cu12>=7.0.0,!=10.1.0'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 3 packages in 27.19s
Prepared 3 packages in 18.18s
Installed 3 packages in 14ms
 + tensorrt-cu12==10.15.1.29
 + tensorrt-cu12-bindings==10.15.1.29
 + tensorrt-cu12-libs==10.15.1.29

requirements: AutoUpdate success ✅ 46.2s
WARNING ⚠️ r